# GraphToken


### 1. 依存関係 & 乱数シード


In [ ]:
import os
import random
import re
from dataclasses import dataclass
from typing import Any

import torch
import torch.nn as nn
from datasets import load_dataset

# PyTorch Geometric
from torch_geometric.data import Batch as PyGBatch
from torch_geometric.data import Data as PyGData
from torch_geometric.nn import GCNConv, global_mean_pool
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # ← 1枚だけ見せる（必ず“最上流”で設定）

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)

### 2. 定数とモデル/トークナイザの準備


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  # 学習済み LLM（凍結）
PROMPT_TOKENS = 16  # グラフ由来“連続プロンプト”の長さ（要調整）
MAX_LENGTH = 2048  # 1サンプルの最大長（テキスト側）
LR = 1e-4  # GNN/FC の学習率（必要なら 3e-5〜1e-4 で調整）

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### 3. データセットの読み込み（Hugging Face Hub）

GraphQA の edge_count サブセットから zero-shot スプリットを読み込みます。


In [ ]:
BASE = "baharef/GraphQA"
DATA_FILES = {
    "train": "edge_count/edge_count_zero_shot_train.json",
    "validation": "edge_count/edge_count_zero_shot_validation.json",
    "test": "edge_count/edge_count_zero_shot_test.json",
}

ds = load_dataset(BASE, data_files=DATA_FILES)
print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['algorithm', 'answer', 'nedges', 'nnodes', 'question', 'task_description', 'text_encoding'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['algorithm', 'answer', 'nedges', 'nnodes', 'question', 'task_description', 'text_encoding'],
        num_rows: 500
    })
    test: Dataset({
        features: ['algorithm', 'answer', 'nedges', 'nnodes', 'question', 'task_description', 'text_encoding'],
        num_rows: 500
    })
})
{'algorithm': 'er', 'answer': ' 14.', 'nedges': '14', 'nnodes': '8', 'question': 'In an undirected graph, (i,j) means that node i and node j are connected with an undirected edge. G describes a graph among nodes 0, 1, 2, 3, 4, 5, 6, and 7.\nThe edges in G are: (0, 1) (0, 5) (0, 6) (0, 7) (1, 2) (1, 4) (1, 5) (1, 6) (1, 7) (2, 3) (2, 4) (3, 7) (4, 5) (5, 7).\nQ: How many edges are in this graph?\nA: ', 'task_description': 'Q: How many edges are in this graph?\nA: ', 'text_encoding': 'adja

In [ ]:
# ★ これを trainer 作成前に定義
from torch.utils.data import Dataset as TorchDataset


class GraphQATorchDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[int(idx)]
        # collator が使うフィールドだけ渡す（必要なら他も）
        return {
            "question": ex["question"],
            "answer": ex["answer"],
        }


train_torch = GraphQATorchDataset(ds["train"])
val_torch = GraphQATorchDataset(ds["validation"])
test_torch = GraphQATorchDataset(ds["test"])

### 4. GraphToken 風アーキテクチャ

- GraphEncoder: 2 層 GCN → global mean pooling → FC（hidden → d_model × P）
- GraphPromptLM: LLM を凍結し、prompt_embeds (B,P,d) をテキスト埋め込みの前に連結して inputs_embeds で前向き。損失は LLM の因果 LM 損失を使用。先頭のグラフトークン分は labels=-100 を前置。


In [ ]:
class GraphEncoder(nn.Module):
    """2層 GCN + mean pool + FC → (B, P, d_model) の連続ソフトプロンプトを生成"""

    def __init__(self, in_dim=1, hidden=256, d_model=1536, prompt_tokens=PROMPT_TOKENS):
        super().__init__()
        self.gcn1 = GCNConv(in_dim, hidden)
        self.gcn2 = GCNConv(hidden, hidden)
        self.proj = nn.Linear(hidden, d_model * prompt_tokens)
        self.prompt_tokens = prompt_tokens
        self.d_model = d_model

    def forward(self, pyg_batch: PyGBatch):
        x = self.gcn1(pyg_batch.x, pyg_batch.edge_index).relu()
        x = self.gcn2(x, pyg_batch.edge_index).relu()
        g = global_mean_pool(x, pyg_batch.batch)  # (B, hidden)
        out = self.proj(g).view(-1, self.prompt_tokens, self.d_model)  # (B, P, d)
        return out


class GraphPromptLM(nn.Module):
    def __init__(self, model_id=MODEL_ID, prompt_tokens=PROMPT_TOKENS):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_id)
        for p in self.llm.parameters():
            p.requires_grad = False
        self.config = self.llm.config  # ★ 追加：TRL が参照する config を公開

        d_model = self.llm.get_input_embeddings().embedding_dim
        self.graph_encoder = GraphEncoder(d_model=d_model, prompt_tokens=prompt_tokens)
        self.embed_tokens = self.llm.get_input_embeddings()
        self.prompt_tokens = prompt_tokens

    def forward(self, input_ids=None, attention_mask=None, labels=None, graphs=None):
        # テキスト側埋め込み（レプリカのdeviceが取れる）
        tok_embeds = self.embed_tokens(input_ids)  # (B_local, T, d)
        device = tok_embeds.device

        # graphs はレプリカごとに List[PyGData] が来る想定（DataParallelがスライス）
        # 互換性のため、Batch/単一Dataが来ても吸収してListに揃える
        if isinstance(graphs, list):
            data_list = [g.to(device) for g in graphs]
        elif hasattr(graphs, "to_data_list"):  # PyG Batch
            data_list = [g.to(device) for g in graphs.to_data_list()]
        else:  # 単一 Data
            data_list = [graphs.to(device)]

        # レプリカ専用 Batch を構築（B_local == len(data_list) を満たす）
        graphs_local = PyGBatch.from_data_list(data_list)

        # グラフ→連続ソフトプロンプト
        prompt_embeds = self.graph_encoder(graphs_local)  # (B_local, P, d)

        # 連結＆ラベル前置 -100
        inputs_embeds = torch.cat([prompt_embeds, tok_embeds], dim=1)
        if labels is not None:
            pad = torch.full((labels.size(0), prompt_embeds.size(1)), -100, dtype=labels.dtype, device=labels.device)
            labels = torch.cat([pad, labels], dim=1)

        return self.llm(inputs_embeds=inputs_embeds, attention_mask=None, labels=labels)

### 5. グラフ抽出ユーティリティ

GraphQA の問題文内から (i,j) 形式のエッジを抽出します。


In [ ]:
edge_pat = re.compile(r"\((\d+),\s*(\d+)\)")


def parse_edges_from_question(q: str):
    edges = [(int(a), int(b)) for (a, b) in edge_pat.findall(q)]
    n_nodes = max((max(u, v) for u, v in edges), default=-1) + 1
    return edges, max(n_nodes, 1)

### 6. completion-only loss を行うカスタム collator

- `apply_chat_template` を用いて、プロンプトのみと プロンプト+アシスタント回答をそれぞれトークナイズ
- `labels = input_ids.clone()` から 先頭 prompt_len を -100 に（assistant のみ損失）
- グラフは PyG の Batch にまとめて graphs として返却


In [ ]:
@dataclass
class GraphCollator:
    tokenizer: Any
    max_length: int = MAX_LENGTH
    graph_node_feat: float = 1.0  # ノード初期特徴（定数 1）

    def __call__(self, batch):
        # --- 1) バッチ形の正規化 ---
        if batch is None:
            raise ValueError("GraphCollator: received None batch")
        if isinstance(batch, dict):  # ← 単一サンプルが dict で来るケース
            batch = [batch]
        if len(batch) == 0:
            raise ValueError("GraphCollator: received empty batch")

        features, graphs = [], []

        for ex in batch:
            if "question" not in ex or "answer" not in ex:
                raise KeyError(
                    f"GraphCollator: missing keys; got {list(ex.keys())}. "
                    "Ensure TrainingArguments(remove_unused_columns=False)."
                )

            q, a = str(ex["question"]).strip(), str(ex["answer"]).strip()

            # --- 2) completion-only: prompt 長の取得 ---
            prompt_only = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": q}],
                tokenize=False,
                add_generation_prompt=True,
            )
            enc_prompt = self.tokenizer(prompt_only, return_tensors="pt", add_special_tokens=False)
            prompt_len = enc_prompt["input_ids"].size(1)

            # --- 3) full（user+assistant）を tokenize ---
            full_text = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": q}, {"role": "assistant", "content": a}],
                tokenize=False,
                add_generation_prompt=False,
            )
            enc_full = self.tokenizer(
                full_text,
                return_tensors="pt",
                add_special_tokens=False,
                truncation=True,
                max_length=self.max_length,
            )
            input_ids = enc_full["input_ids"][0]
            attention_mask = enc_full["attention_mask"][0]

            labels = input_ids.clone()
            labels[: min(prompt_len, labels.size(0))] = -100  # completion-only

            # ★ ここがポイント：pad に渡す前に list[int] 化
            features.append(
                {
                    "input_ids": input_ids.tolist(),
                    "attention_mask": attention_mask.tolist(),
                    "labels": labels.tolist(),
                }
            )

            # --- 4) グラフ抽出 → PyG Data ---
            edges, n = parse_edges_from_question(q)
            n = max(int(n), 1)
            x = torch.ones((n, 1), dtype=torch.float32) * self.graph_node_feat
            if len(edges) > 0:
                edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            else:
                edge_index = torch.empty((2, 0), dtype=torch.long)
            graphs.append(PyGData(x=x, edge_index=edge_index))

        # --- 5) テキスト側のパディング（input_ids/attention_mask のみ tokenizer.pad に任せる） ---
        inputs_for_pad = [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features]
        padded_inputs = self.tokenizer.pad(
            inputs_for_pad,
            padding=True,  # or padding="max_length" + max_length=self.max_length
            return_tensors="pt",
        )

        # --- 5') labels は手動でパディング（-100 で右詰め） ---
        max_len = padded_inputs["input_ids"].size(1)
        padded_labels = []
        for f in features:
            lab = f["labels"]
            if len(lab) > max_len:
                # 念のため（通常は enc_full で truncation 済みなので入らない）
                lab = lab[:max_len]
            if len(lab) < max_len:
                lab = lab + ([-100] * (max_len - len(lab)))
            padded_labels.append(lab)
        padded_labels = torch.tensor(padded_labels, dtype=torch.long)

        # --- 6) グラフ側のバッチ ---
        if len(graphs) == 0:
            graphs = [
                PyGData(x=torch.ones((1, 1), dtype=torch.float32), edge_index=torch.empty((2, 0), dtype=torch.long))
            ]
        # pyg_batch = PyGBatch.from_data_list(graphs)

        return {
            "input_ids": padded_inputs["input_ids"],
            "attention_mask": padded_inputs["attention_mask"],
            "labels": padded_labels,
            "graphs": graphs,  # ✅ List[PyGData] のまま返す
        }


collator = GraphCollator(tokenizer=tokenizer, max_length=MAX_LENGTH)

In [17]:
from torch.utils.data import DataLoader

dl = DataLoader(train_torch, batch_size=1, shuffle=False, collate_fn=collator)
first_batch = next(iter(dl))
for k, v in first_batch.items():
    print(k, type(v), getattr(v, "shape", None))
# ここで "graphs" が PyG Batch になっているか、input_ids/labels の形が期待どおりか確認

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


input_ids <class 'torch.Tensor'> torch.Size([1, 175])
attention_mask <class 'torch.Tensor'> torch.Size([1, 175])
labels <class 'torch.Tensor'> torch.Size([1, 175])
graphs <class 'list'> None


### 7. Trainer 準備 & 学習

SFTTrainer は最新の TRL では `tokenizer=` 引数を取りません。代わりに `processing_class=tokenizer` を渡します。


In [ ]:
from transformers import Trainer, TrainingArguments

# model = GraphPromptLM(MODEL_ID, PROMPT_TOKENS)
model = GraphPromptLM(MODEL_ID, PROMPT_TOKENS).to("cuda")


training_args = TrainingArguments(
    output_dir="glm-graphprompt-qwen3-4b",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LR,
    num_train_epochs=1,
    logging_steps=20,
    save_steps=1000,
    bf16=True,
    fp16=False,
    max_grad_norm=1.0,
    report_to=[],
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_torch,  # ← Torch Dataset ラップ
    eval_dataset=val_torch,
    data_collator=collator,  # ← カスタム collator（completion-only + graph）
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.41it/s]


RuntimeError: Caught RuntimeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/naoki/anaconda3/envs/g-retriever/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py", line 84, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/naoki/anaconda3/envs/g-retriever/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/naoki/anaconda3/envs/g-retriever/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_165293/1680969896.py", line 54, in forward
    inputs_embeds = torch.cat([prompt_embeds, tok_embeds], dim=1)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: Sizes of tensors must match except i

### 8. 推論ヘルパ & 簡易評価（zero_shot_test）

- 推論では inputs_embeds を使って グラフ由来プロンプト + ユーザープロンプトを連結
- 単純な数値一致で精度を概算（edge_count の答えは整数）


In [ ]:
num_pat = re.compile(r"(-?\d+)")


def generate_answer(model: GraphPromptLM, question: str, max_new_tokens: int = 32):
    edges, n = parse_edges_from_question(question)
    x = torch.ones((n, 1), dtype=torch.float32)
    edge_index = (
        torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2, 0), dtype=torch.long)
    )
    pyg_batch = PyGBatch.from_data_list([PyGData(x=x, edge_index=edge_index)])

    messages = [{"role": "user", "content": question.strip()}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tok = tokenizer(prompt, return_tensors="pt")

    model.eval()
    with torch.no_grad():
        tok_embeds = model.embed_tokens(tok["input_ids"])  # (1, T, d)
        gprompt = model.graph_encoder(pyg_batch)  # (1, P, d)
        inputs_embeds = torch.cat([gprompt, tok_embeds], dim=1)  # (1, P+T, d)
        out_ids = model.llm.generate(inputs_embeds=inputs_embeds, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)


# 簡易 Accuracy（最初の整数同士で比較）
correct = 0
for ex in ds["test"]:
    pred = generate_answer(model, ex["question"])
    tp = num_pat.findall(pred)
    ta = num_pat.findall(ex["answer"])
    if tp and ta and tp[0] == ta[0]:
        correct += 1
acc = correct / len(ds["test"]) if len(ds["test"]) else float("nan")
print(f"Test Accuracy: {acc:.3f}")

### 9. 保存（学習した GNN ＋ FC のみ）


In [ ]:
# LLM は凍結のままなので、graph_encoder の重みだけを保存する例
os.makedirs("glm-graphprompt-qwen3-4b/artifacts", exist_ok=True)
torch.save(model.graph_encoder.state_dict(), "glm-graphprompt-qwen3-4b/artifacts/graph_encoder.pt")